# sentiment-scale walkthrough

A short tour of the pure-numpy/pandas core. We:

1. run the seeded end-to-end demo (`run_demo`);
2. compare the **lexicon** scorer against a **trained logistic-regression**
   model on the same posts;
3. attach a **bootstrap confidence interval** to the mean sentiment;
4. pull out **NMF topics** from the corpus.

Everything here is pure numpy/pandas/stdlib — no Spark, no transformers,
no vaderSentiment. Set `PYTHONPATH=src` (or `pip install -e .`) first.


## 1. The seeded demo

`run_demo(0)` synthesises labelled posts, scores them with the lexicon,
validates the sign against the known labels (accuracy 1.0), and recovers a
planted sentiment shift. The numbers are deterministic.


In [ ]:
from sentiment.demo import run_demo

result = run_demo(0, out_dir="../outputs")
{k: result[k] for k in (
    "num_posts", "validation_accuracy",
    "mean_sentiment_before", "mean_sentiment_after", "shift_date",
)}


## 2. Lexicon vs trained model

The lexicon is a fixed valence table. The `LogisticRegression` classifier is
*learned* from labelled bag-of-words features. On a small, linearly separable
labelled set it reaches 100% train accuracy and agrees with the lexicon's sign
on clear cases — but it is trained, not hand-specified.


In [ ]:
import numpy as np
from sentiment.lexicon import score_text
from sentiment.classify import LogisticRegression, bag_of_words

train_docs = [
    "great good wonderful", "good great good", "wonderful great good",
    "bad terrible awful", "terrible bad awful", "awful terrible bad",
]
y = np.array([1, 1, 1, 0, 0, 0])

X, vocab = bag_of_words(train_docs)
clf = LogisticRegression(lr=0.5, n_iters=2000).fit(X, y)
train_acc = float((clf.predict(X) == y).mean())

# Score a couple of fresh sentences with BOTH approaches.
tests = ["great good", "terrible awful"]
Xt, _ = bag_of_words(tests, vocab=vocab)
for text, p in zip(tests, clf.predict_proba(Xt)):
    print(f"{text!r:24s} lexicon={score_text(text):+.3f}  model P(pos)={p:.3f}")
print("trained-model train accuracy:", train_acc)


## 3. Uncertainty: bootstrap CI for the mean

A weekly mean is a point estimate. `bootstrap_mean_ci` attaches a 95%
percentile-bootstrap interval so we quote a band, not a bare number.


In [ ]:
from sentiment.uncertainty import bootstrap_mean_ci

scores = result_scores = [
    score_text(t) for t in train_docs
]
lo, hi = bootstrap_mean_ci(scores, n_boot=2000, seed=0, alpha=0.05)
print(f"mean = {np.mean(scores):+.3f}   95% CI = [{lo:+.3f}, {hi:+.3f}]")


## 4. NMF topics

`nmf` factorises a non-negative TF-IDF (or bag-of-words) matrix into
non-negative document-topic (`W`) and topic-term (`H`) factors. The top terms of
each row of `H` read as a topic.


In [ ]:
from sentiment.topics import tfidf, nmf

docs = [
    "good great wonderful", "great good good",
    "bad terrible awful", "terrible bad awful",
]
matrix, vocab = tfidf(docs)
W, H = nmf(matrix, k=2, iters=500, seed=0)

for topic_idx, row in enumerate(H):
    top = [vocab[j] for j in row.argsort()[::-1][:3]]
    print(f"topic {topic_idx}: {top}")
print("factors non-negative:", bool((W >= 0).all() and (H >= 0).all()))


## Takeaways

- The lexicon and the trained model agree on clear cases; the model is the
  learned alternative you validate against where they disagree.
- Always carry an uncertainty band (bootstrap CI) on an aggregated mean.
- NMF gives transparent, non-negative topics to read alongside the sentiment
  series — a sentiment move that lines up with a topic move is a topic effect.

See `README.md` and `USAGE.md` for the honest list of what these scores do and
do not support.
